**Whisper large-v3 (chunk)**: CSE+морфология (3a дәптерінен), BERT+морфология+MaxEnt, BERT+морфология+CRF.

Алдымен `MAPP_3a_lexical_largev3.ipynb` аяқталуы керек (ол `largev3_lex/…_lex.csv` жасайды). Негізгі мақсат — MaxEnt нәтижесі (stage 3); CRF-ті де жүргізуге болады (checkpoint бар). Нәтиже `largev3_lex/` қалтасына жазылады; соңғы cell `mapp_unified_texts_largev3.csv` жасайды.

# MAPP-ASR 2-саты (large-v3+chunk RAW негізінде): BERT қате-детекторы + морфология-аналитикалық кандидат генерация + MaxEnt reranker

Бұл notebook — Stage 2 (BERT+MaxEnt) каскадының, ASR деңгейі жақсартылған
(**Whisper large-v3 + 30с chunking + anti-hallucination decoding**) RAW
транскриптерге қолданылған нұсқасы.

Кіріс: `mapp_v2_batch_results_asr_upgrade.csv` (large-v3+chunk RAW/лексикалық-FIXED,
ASR-upgrade экспериментінен).

Шығыс: `mapp_stage2_on_largev3_comparison.xlsx` — RAW (large-v3) vs лексикалық MAPP v2
(large-v3 негізінде) vs BERT+MaxEnt (large-v3 негізінде) салыстыруы.

⚠️ Барлық жоғарыдағы cell-дерді (корпус жүктеу, синтетикалық қате генерациясы,
BERT детектор оқыту, MaxEnt reranker оқыту) бұрынғыдай қалдырамыз — тек нақты
100-аудио блогындағы кіріс файлды ауыстырамыз.


In [31]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH"
import os
os.makedirs(BASE_DIR, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH


### Diagnostics: қазақ корпусы қай жерде?

Төмендегі cell `BASE_DIR` ішінен .txt/.csv/.tsv файлдарды іздейді және әрқайсысының
алғашқы 2 жолын көрсетеді — қай файл нағыз қазақ корпусы екенін көзбен растау үшін.
Егер ештеңе табылмаса, cell сізге корпусты қайда салу керектігін нақты айтады және
жалғастырмайды (келесі cell-дер соны талап етеді).

In [32]:
import glob

candidate_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "*.txt")) +
    glob.glob(os.path.join(BASE_DIR, "*.csv")) +
    glob.glob(os.path.join(BASE_DIR, "*.tsv"))
)

print(f"{BASE_DIR} ішінде табылған файлдар: {len(candidate_files)}")
for fp in candidate_files:
    size_mb = os.path.getsize(fp) / 1e6
    print(f"\n--- {fp}  ({size_mb:.2f} MB) ---")
    try:
        with open(fp, encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= 2:
                    break
                print("  ", line.strip()[:120])
    except Exception as e:
        print("  [оқу қатесі]", e)

if not candidate_files:
    print(
        "\n[ЕСКЕРТУ] Бұл қалтада корпус файлы табылмады.\n"
        f"'{BASE_DIR}' қалтасына таза қазақ мәтін корпусын салыңыз "
        "(жол сайын бір сөйлем, .txt; немесе 'kazakh'/'kk'/'text' атты\n"
        "бағаны бар .csv/.tsv). Мыс.: kazakh_285000.txt, leipzig_kaz_news.txt, "
        "т.б. Содан кейін бұл cell-ді қайта іске қосыңыз."
    )

/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH ішінде табылған файлдар: 5

--- /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/corpus_kaz_285000.txt  (68.57 MB) ---
   kazakh
   соңғы бес жылда ана өлімі шамамен 3 есе азайды , бала туу көрсеткіші бір жарым есе өсті .

--- /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/kazakh_285000.txt  (68.57 MB) ---
   kazakh
   соңғы бес жылда ана өлімі шамамен 3 есе азайды , бала туу көрсеткіші бір жарым есе өсті .

--- /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/kazakh_497000_corpus.txt  (82.65 MB) ---
   Жүрек бар жерде арман бар!
   Нәтижелер Мейірбикелік күтімнің негізгі әдістері пациенттерге психиатриялық көмек көрсету үшін медбикелердің іргелі диат

--- /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/kazakh_noisy_correct_pairs.csv  (65.48 MB) ---
   noisy_sentence,correct_sentence,labels
   соңғы бес жылда ана өлімі шамамен 3 есе азайди бала туу көрсеткіші бір жарым есе өсти,соңғы бес жылда ана өлімі ш

### Корпусты жүктеу

`CORPUS_PATH`-ты жоғарыдағы тізімнен өзіңіз таңдаңыз (ең үлкен/сапалы файлды),
немесе бос қалдырсаңыз бірінші табылған файл автоматты алынады.

In [33]:
import re
import pandas as pd

# Алдымен нақты атауды тексереміз, ол табылмаса (мыс. Drive-та файл аты
# басқаша болса), жоғарыдағы diagnostics cell-де табылған алғашқы корпус
# файлын (candidate_files[0]) автоматты түрде қолданамыз.
_CORPUS_GUESS = os.path.join(BASE_DIR, "corpus_kaz_285000.txt")
if os.path.exists(_CORPUS_GUESS):
    CORPUS_PATH = _CORPUS_GUESS
elif "candidate_files" in dir() and candidate_files:
    CORPUS_PATH = candidate_files[0]
    print(f"[АҚПАРАТ] '{_CORPUS_GUESS}' табылмады, орнына табылған бірінші "
          f"файл қолданылады: {CORPUS_PATH}")
else:
    raise FileNotFoundError(
        f"Корпус файлы табылмады: '{_CORPUS_GUESS}' де жоқ, "
        f"'{BASE_DIR}' қалтасында да .txt/.csv/.tsv файл жоқ. "
        "Жоғарыдағы diagnostics cell-ді (candidate_files) қайта іске қосыңыз "
        "немесе корпус файлын сол қалтаға жүктеңіз."
    )

def clean_kazakh_sentence(s):
    """
    Тазалау + маңызды түзету: корпустарда тыныс белгілері ЕКІ түрлі форматта
    кездеседі — 285 мыңдық ескі корпуста БӨЛЕК токен ретінде ("есе , бала"),
    ал жаңа қосылған (497 мыңдық) корпуста сөзге ЖАБЫСЫП тұр ("сөз,", "сөз."
    — нақты Whisper аудио-мәтінімен бірдей форматта). Екі жағдайда да сөздікте
    таза "өзек" сөз қалуы керек (тыныс белгісі жоқ), сондықтан:
      1) бүтіндей тыныс-белгі токендерін алып тастаймыз (ескі корпус үшін),
      2) қалған әр токеннің БАСЫНДА/СОҢЫНДА тұрған тыныс белгілерін кесіп
         тастаймыз (жаңа корпус үшін) — "сөз," -> "сөз", "«сөз»" -> "сөз".
    Нақты аудио мәтінін өңдеу кезінде де сөздің тыныс белгісін алдын ала
    бөліп алып, соңынан қайта жабыстырамыз (төмендегі `correct_sentence_real`
    функциясын қараңыз), сондықтан бұл екі формат осылай сәйкестендіріледі.
    """
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    tokens = []
    for t in s.split(" "):
        if not any(ch.isalnum() for ch in t):
            continue  # бүтіндей тыныс-белгі токені — тастап кетеміз
        # токеннің басы/соңындағы тыныс белгілерін кесеміз, ортасындағы
        # апострофты/дефисті (сөздің құрамдас бөлігі болуы мүмкін) сақтаймыз
        t_clean = re.sub(r"^[^\w]+|[^\w]+$", "", t, flags=re.UNICODE)
        if t_clean:
            tokens.append(t_clean)
    return " ".join(tokens)


def _cyrillic_ratio(s):
    """Жолдағы әріптердің қаншасы кириллица екенін есептейді (0..1)."""
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return 0.0
    cyr = sum(1 for c in letters if "\u0400" <= c <= "\u04FF")
    return cyr / len(letters)


def is_kazakh_like_sentence(s, min_cyrillic_ratio=0.5):
    """
    Бөгде тілдегі/junk жолдарды (мыс. компания атаулары ағылшынша,
    "Changzhou Huatai ... Co. Ltd.") сүзу үшін: кириллица әріптерінің
    үлесі min_cyrillic_ratio-дан төмен болса, жол тасталады.
    """
    return _cyrillic_ratio(s) >= min_cyrillic_ratio


def _load_one_corpus_file(path):
    """Бір корпус файлын (TXT/CSV/TSV) оқып, сөйлемдер тізімін қайтарады."""
    path = str(path)
    if path.lower().endswith((".tsv", ".csv")):
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep)
        col = None
        for c in ["kazakh", "kk", "text"]:
            if c in df.columns:
                col = c
                break
        if col is None:
            raise ValueError(f"Қазақ бағаны табылмады {path}. Бағандар: {list(df.columns)}")
        sentences = df[col].dropna().astype(str).tolist()
    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            sentences = f.read().splitlines()
    return sentences


def load_kazakh_corpus(path=None, max_sentences=None, merge_all_found=True):
    """
    Таза қазақ корпусын жүктейді.
    Қолдау көрсетілетін форматтар:
      - TXT: жол сайын бір сөйлем
      - TSV/CSV: 'kazakh' / 'kk' / 'text' атты баған

    merge_all_found=True болса (әдепкі), BASE_DIR ішінде diagnostics
    cell-де табылған БАРЛЫҚ корпус файлдарын (candidate_files) біріктіріп
    жүктейді — мыс. бастапқы 285 мың сөйлемдік файлмен қатар жаңа қосылған
    497 мың сөйлемдік файл да бар болса, екеуі де қосылады, қайталанатын
    сөйлемдер (dedupe) алынып тасталады.
    """
    global candidate_files
    all_sentences = []

    if merge_all_found and "candidate_files" in dir() and candidate_files:
        for fp in candidate_files:
            try:
                s = _load_one_corpus_file(fp)
                print(f"  + {fp}: {len(s)} жол")
                all_sentences.extend(s)
            except Exception as e:
                print(f"  [өткізіп жіберілді] {fp}: {e}")
    else:
        if path is None:
            if not candidate_files:
                raise FileNotFoundError(
                    "Қазақ корпусы табылмады. Жоғарыдағы diagnostics cell-дегі "
                    "нұсқауды қараңыз: корпус файлын BASE_DIR қалтасына салыңыз."
                )
            path = candidate_files[0]
        all_sentences = _load_one_corpus_file(path)

    before_lang_filter = len(all_sentences)
    all_sentences = [s for s in all_sentences if is_kazakh_like_sentence(s)]
    dropped_lang = before_lang_filter - len(all_sentences)
    if dropped_lang:
        print(f"  [сүзілді] {dropped_lang} бөгде тіл/junk жол алынып тасталды "
              f"(кириллица үлесі <50%)")

    sentences = [clean_kazakh_sentence(s) for s in all_sentences]
    sentences = [s for s in sentences if len(s.split()) >= 3]

    before_dedupe = len(sentences)
    seen = set()
    deduped = []
    for s in sentences:
        if s not in seen:
            seen.add(s)
            deduped.append(s)
    sentences = deduped
    print(f"Дедупликациядан бұрын: {before_dedupe}, кейін: {len(sentences)}")

    if max_sentences:
        sentences = sentences[:max_sentences]

    print("Жүктелген сөйлем саны (жиынтық):", len(sentences))
    return sentences


clean_sentences = load_kazakh_corpus(CORPUS_PATH, max_sentences=None, merge_all_found=True)
print(clean_sentences[:3])

  [сүзілді] 1846 бөгде тіл/junk жол алынып тасталды (кириллица үлесі <50%)
Дедупликациядан бұрын: 282191, кейін: 282189
Жүктелген сөйлем саны (жиынтық): 282189
['соңғы бес жылда ана өлімі шамамен 3 есе азайды бала туу көрсеткіші бір жарым есе өсті', 'ұлы қолбасшы сұлтан бейбарыс қазақ даласында дүниега келіп тағдырдың жазуымен араб жерінде оған берілген миссияны азаматтықпен және абыроймен орындады', 'шағын ауданды аралау кезінде облыс әкімі қ қожамжаров мемлекет басшысын президенттің тапсырмасы бойынша әзірленген 2011-2014 жылдарға арналған астана қаласына іргелес елді мекендерді дамытудың кешенді жоспарын ісқа асыру барысымен таныстырды']


In [34]:
!pip -q install transformers datasets scikit-learn pandas numpy torch evaluate jiwer sacrebleu bert_score rapidfuzz

In [35]:
import random, math, json, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path(BASE_DIR)
DATA_DIR.mkdir(exist_ok=True, parents=True)
print(DATA_DIR)

/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH


## Синтетикалық ASR-қате генераторы (қазақ фонология/морфологиясы негізінде)

Бұл ережелер жоғарыда расталғандай нағыз қазақ-спецификалық (ә↔а, ө↔о, ү/ұ↔у,
қ↔к, ғ↔г, ң↔н, і↔и, септік/тәуелдік жалғауларының ASR-де жиі шатасатын түрлері).

In [36]:
CHAR_RULES = [
    ('ә', 'а'), ('ө', 'о'), ('ү', 'у'), ('ұ', 'у'),
    ('қ', 'к'), ('ғ', 'г'), ('ң', 'н'), ('һ', 'х'),
    ('і', 'и'),
    ('Ә', 'А'), ('Ө', 'О'), ('Ү', 'У'), ('Ұ', 'У'),
    ('Қ', 'К'), ('Ғ', 'Г'), ('Ң', 'Н'), ('Һ', 'Х'),
    ('І', 'И'),
]

VOWEL_REDUCTION_RULES = [
    ('ы', ''), ('і', ''), ('а', ''), ('е', '')
]

SUFFIX_RULES = [
    ('ның', 'нын'), ('нің', 'нин'),
    ('дың', 'дын'), ('дің', 'дин'),
    ('тың', 'тын'), ('тің', 'тин'),
    ('ға', 'га'), ('қа', 'ка'),
    ('ды', 'ди'), ('ді', 'ди'),
    ('ты', 'ти'), ('ті', 'ти'),
    ('бен', 'пен'), ('пен', 'бен'),
    ('дар', 'тар'), ('дер', 'тер'),
    ('тар', 'дар'), ('тер', 'дер'),
    ('ған', 'ган'), ('қан', 'кан'),
    ('мын', 'мин'), ('мін', 'мин'),
    ('сың', 'сын'), ('сің', 'син'),
]

DOUBLE_RULES = [
    ('кк', 'к'), ('тт', 'т'), ('пп', 'п'),
    ('лл', 'л'), ('мм', 'м'), ('нн', 'н'),
]


def corrupt_token(token, p_char=0.28, p_vowel_drop=0.12,
                   p_suffix=0.45, p_double=0.20):
    original = token
    noisy = token

    if random.random() < p_suffix:
        applicable = [(a, b) for a, b in SUFFIX_RULES if noisy.endswith(a) and a != b]
        if applicable:
            a, b = random.choice(applicable)
            noisy = noisy[:-len(a)] + b

    if random.random() < p_char:
        applicable = [(a, b) for a, b in CHAR_RULES if a in noisy]
        if applicable:
            a, b = random.choice(applicable)
            noisy = noisy.replace(a, b, 1)

    if random.random() < p_vowel_drop and len(noisy) > 4:
        applicable = [(a, b) for a, b in VOWEL_REDUCTION_RULES if a in noisy[1:-1]]
        if applicable:
            a, b = random.choice(applicable)
            pos = noisy.find(a, 1)
            if 0 < pos < len(noisy) - 1:
                noisy = noisy[:pos] + b + noisy[pos + len(a):]

    if random.random() < p_double:
        applicable = [(a, b) for a, b in DOUBLE_RULES if a in noisy]
        if applicable:
            a, b = random.choice(applicable)
            noisy = noisy.replace(a, b, 1)

    return noisy, noisy != original


def corrupt_sentence(sentence, token_error_prob=0.35):
    tokens = sentence.split()
    noisy_tokens, labels = [], []

    for tok in tokens:
        if random.random() < token_error_prob:
            noisy_tok, changed = corrupt_token(tok)
        else:
            noisy_tok, changed = tok, False

        noisy_tokens.append(noisy_tok)
        labels.append('ERR' if changed else 'OK')

    return ' '.join(noisy_tokens), labels


for s in clean_sentences[:5]:
    noisy, labels = corrupt_sentence(s)
    print('CLEAN:', s)
    print('NOISY:', noisy)
    print('LAB  :', labels)
    print()

CLEAN: соңғы бес жылда ана өлімі шамамен 3 есе азайды бала туу көрсеткіші бір жарым есе өсті
NOISY: соңғы бес жылда ана өлімі шамамен 3 есе азайды бала туу көрсеткіші бір жарым есе өсті
LAB  : ['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

CLEAN: ұлы қолбасшы сұлтан бейбарыс қазақ даласында дүниега келіп тағдырдың жазуымен араб жерінде оған берілген миссияны азаматтықпен және абыроймен орындады
NOISY: ұлы қолбасшы сұлтан бейбарыс қазақ даласнда дүниега келіп тағдырдың жазуымен араб жерінде оған берілген миссияны азаматтықпен және абыроймен орындады
LAB  : ['OK', 'OK', 'OK', 'OK', 'OK', 'ERR', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

CLEAN: шағын ауданды аралау кезінде облыс әкімі қ қожамжаров мемлекет басшысын президенттің тапсырмасы бойынша әзірленген 2011-2014 жылдарға арналған астана қаласына іргелес елді мекендерді дамытудың кешенді жоспарын ісқа асыру барысымен таныстырды
NOISY: шағын ауданды 

## Noisy/correct жұптарын құру

`N_SENTENCES`-ті Colab уақыт шектеуіне қарай реттеңіз (толық корпус GPU-мен де
бірнеше сағатқа созылуы мүмкін; алдымен 30–50 мыңмен сынап көріңіз).

In [37]:
# Корпус 285 мыннан ~780 мыңға дейін өскендіктен, оқыту үлгісін де
# көбейтеміз (әйтпесе жаңа сөйлемдер тек word_freq/vocab-қа әсер етіп,
# BERT/MaxEnt оқыту деректеріне кірмей қалар еді). 150 мың — GPU уақыты
# мен деректің көбеюінің арасындағы орынды теңгерім (~3x бұрынғыдан);
# уақытыңыз жетсе, бұл санды одан әрі көбейтуге болады.
N_SENTENCES = min(len(clean_sentences), 150_000)  # қажет болса көбейтіңіз/азайтыңыз
print(f"Оқытуға пайдаланылатын сөйлем саны: {N_SENTENCES} (қолда бар корпус: {len(clean_sentences)})")

rows = []
for sent in clean_sentences[:N_SENTENCES]:
    noisy, labels = corrupt_sentence(sent, token_error_prob=0.35)
    if noisy != sent:
        rows.append({
            'noisy_sentence': noisy,
            'correct_sentence': sent,
            'labels': ' '.join(labels)
        })
print("rows:", len(rows))
pairs_df = pd.DataFrame(rows)
print(pairs_df.shape)
pairs_df.head()

pairs_path = DATA_DIR / 'kazakh_noisy_correct_pairs.csv'
pairs_df.to_csv(pairs_path, index=False, encoding='utf-8')
print('Сақталды:', pairs_path)

Оқытуға пайдаланылатын сөйлем саны: 150000 (қолда бар корпус: 282189)
rows: 116578
(116578, 3)
Сақталды: /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/kazakh_noisy_correct_pairs.csv


In [38]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(pairs_df, test_size=0.10, random_state=SEED)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

print('train:', len(train_df), 'valid:', len(valid_df), 'test:', len(test_df))

train: 104920 valid: 5829 test: 5829


In [39]:
from datasets import Dataset, DatasetDict

def to_hf(df):
    return Dataset.from_pandas(pd.DataFrame({
        'tokens': df['noisy_sentence'].apply(str.split),
        'labels_str': df['labels'].apply(str.split),
        'correct_tokens': df['correct_sentence'].apply(str.split),
    }))

hf_data = DatasetDict({
    'train': to_hf(train_df),
    'validation': to_hf(valid_df),
    'test': to_hf(test_df),
})

hf_data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels_str', 'correct_tokens', '__index_level_0__'],
        num_rows: 104920
    })
    validation: Dataset({
        features: ['tokens', 'labels_str', 'correct_tokens', '__index_level_0__'],
        num_rows: 5829
    })
    test: Dataset({
        features: ['tokens', 'labels_str', 'correct_tokens', '__index_level_0__'],
        num_rows: 5829
    })
})

In [40]:
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {'OK': 0, 'ERR': 1}
id2label = {0: 'OK', 1: 'ERR'}

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    all_labels = []
    for i, labels in enumerate(examples['labels_str']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(label2id[labels[word_id]])
            else:
                label_ids.append(-100)
            prev_word_id = word_id

        all_labels.append(label_ids)

    tokenized['labels'] = all_labels
    return tokenized

tokenized_data = hf_data.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=hf_data['train'].column_names
)

tokenized_data

Map:   0%|          | 0/104920 [00:00<?, ? examples/s]

Map:   0%|          | 0/5829 [00:00<?, ? examples/s]

Map:   0%|          | 0/5829 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 104920
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5829
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5829
    })
})

In [41]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

MAX_TRAIN_SAMPLES = 20000   # толық корпус үшін None қойыңыз
MAX_EVAL_SAMPLES = 3000

train_tok = tokenized_data['train']
valid_tok = tokenized_data['validation']

if MAX_TRAIN_SAMPLES:
    train_tok = train_tok.select(range(min(MAX_TRAIN_SAMPLES, len(train_tok))))
if MAX_EVAL_SAMPLES:
    valid_tok = valid_tok.select(range(min(MAX_EVAL_SAMPLES, len(valid_tok))))

_SAVED_DETECTOR = str(DATA_DIR / 'bert_ok_err' / 'final')
data_collator = DataCollatorForTokenClassification(tokenizer)

if os.path.isdir(_SAVED_DETECTOR) and os.path.exists(os.path.join(_SAVED_DETECTOR, 'config.json')):
    # -----------------------------------------------------------
    # Бұл детектор СИНТЕТИКАЛЫҚ корпус-негізді қате үлгісінде
    # оқытылған (нақты RAW-ға тәуелсіз), сондықтан asr_upgrade
    # үшін бұрын оқытылған детекторды bigcorpus RAW-да да қауіпсіз
    # қайта пайдалануға болады -- қайта оқытпаймыз.
    # -----------------------------------------------------------
    print(f"[АҚПАРАТ] Дайын BERT детектор табылды: {_SAVED_DETECTOR}")
    print("Қайта оқытылмайды, тікелей жүктеледі.")
    bert_detector = AutoModelForTokenClassification.from_pretrained(_SAVED_DETECTOR)
    tokenizer = AutoTokenizer.from_pretrained(_SAVED_DETECTOR)
    data_collator = DataCollatorForTokenClassification(tokenizer)
else:
    print(f"[АҚПАРАТ] {_SAVED_DETECTOR} табылмады -- нөлден оқытамыз (CPU-да ~5-6 сағ).")
    bert_detector = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        true_preds, true_labels = [], []
        for p_seq, l_seq in zip(preds, labels):
            for p, l in zip(p_seq, l_seq):
                if l != -100:
                    true_preds.append(p)
                    true_labels.append(l)

        true_preds = np.array(true_preds)
        true_labels = np.array(true_labels)

        acc = (true_preds == true_labels).mean()
        err_mask = true_labels == 1
        err_recall = (true_preds[err_mask] == 1).mean() if err_mask.any() else 0.0

        return {'token_accuracy': acc, 'err_recall': err_recall}

    args = TrainingArguments(
        output_dir=str(DATA_DIR / 'bert_ok_err'),
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        eval_strategy='steps',
        eval_steps=500,
        save_steps=500,
        logging_steps=100,
        report_to='none'
    )

    trainer = Trainer(
        model=bert_detector,
        args=args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    _ckpt_dir = str(DATA_DIR / 'bert_ok_err')
    _has_ckpt = os.path.isdir(_ckpt_dir) and any(p.startswith("checkpoint-") for p in os.listdir(_ckpt_dir))
    trainer.train(resume_from_checkpoint=True) if _has_ckpt else trainer.train()

    trainer.save_model(_SAVED_DETECTOR)
    tokenizer.save_pretrained(_SAVED_DETECTOR)


[АҚПАРАТ] Дайын BERT детектор табылды: /content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/bert_ok_err/final
Қайта оқытылмайды, тікелей жүктеледі.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Морфология-аналитикалық кандидат генерациясы

In [42]:
word_freq = Counter()
for s in clean_sentences:
    word_freq.update(s.split())

VOCAB = set(word_freq.keys())
print('Vocab size:', len(VOCAB))

REVERSE_RULES = [
    ('а', 'ә'), ('о', 'ө'), ('у', 'ү'), ('у', 'ұ'),
    ('к', 'қ'), ('г', 'ғ'), ('н', 'ң'), ('х', 'һ'),
    ('и', 'і'),

    ('нын', 'ның'), ('нин', 'нің'),
    ('дын', 'дың'), ('дин', 'дің'),
    ('тын', 'тың'), ('тин', 'тің'),

    ('га', 'ға'), ('ка', 'қа'),

    ('ди', 'ды'), ('ди', 'ді'),
    ('ти', 'ты'), ('ти', 'ті'),

    ('ган', 'ған'), ('кан', 'қан'),
    ('мин', 'мын'), ('мин', 'мін'),
    ('сын', 'сың'), ('син', 'сің'),

    ('тар', 'дар'), ('тер', 'дер'),
    ('дар', 'тар'), ('дер', 'тер'),
]

def generate_candidates(token, max_candidates=30, use_vocab=True):
    cands = {token}

    for bad, good in REVERSE_RULES:
        if token.endswith(bad) and bad:
            cands.add(token[:-len(bad)] + good)

    for bad, good in REVERSE_RULES:
        if bad and bad in token:
            cands.add(token.replace(bad, good, 1))

    if len(token) >= 3:
        for pos in range(1, len(token)):
            for ch in ['ы', 'і', 'а', 'е']:
                cands.add(token[:pos] + ch + token[pos:])

    if use_vocab:
        vocab_cands = {c for c in cands if c in VOCAB or c == token}
        if vocab_cands:
            cands = vocab_cands | {token}

    cands = list(cands)
    cands.sort(
        key=lambda w: (
            int(w in VOCAB),
            word_freq.get(w, 0),
            -abs(len(w) - len(token))
        ),
        reverse=True
    )
    return cands[:max_candidates]

print(generate_candidates('мектептын'))
print(generate_candidates('оқушылары'))
print(generate_candidates('сабака'))

Vocab size: 183950
['мектептың', 'мектептын']
['оқушылары']
['сабака']


In [43]:
def kazakh_vowel_harmony_ok(token):
    front = set('әөүіе')
    back = set('аоұы')
    t = token.lower()
    vowels = [ch for ch in t if ch in front or ch in back]
    if not vowels:
        return True

    has_front = any(ch in front for ch in vowels)
    has_back = any(ch in back for ch in vowels)

    return not (has_front and has_back)


def edit_distance(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[-1][-1]


KAZAKH_SUFFIXES = [
    'ның', 'нің', 'дың', 'дің', 'тың', 'тің',
    'ға', 'ге', 'қа', 'ке',
    'ды', 'ді', 'ты', 'ті',
    'лар', 'лер', 'дар', 'дер', 'тар', 'тер',
    'мен', 'бен', 'пен',
    'ған', 'ген', 'қан', 'кен',
    'мын', 'мін', 'бын', 'бін', 'пын', 'пін',
    'сың', 'сің'
]

MOTION_WORDS = {
    'барды', 'келді', 'кетті', 'кірді', 'шықты',
    'барады', 'келеді', 'кетеді', 'кіреді', 'шығады'
}


def maxent_features(src, cand, left='', right=''):
    feats = {}

    feats['bias'] = 1
    feats['copy'] = int(src == cand)
    feats['change'] = int(src != cand)
    feats['edit_distance'] = edit_distance(src, cand)
    feats['len_diff'] = abs(len(src) - len(cand))
    feats['cand_in_vocab'] = int(cand in VOCAB)
    feats['log_freq'] = math.log1p(word_freq.get(cand, 0))
    feats['vowel_harmony_ok'] = int(kazakh_vowel_harmony_ok(cand))

    feats['restore_a_to_ae'] = int('а' in src and 'ә' in cand)
    feats['restore_o_to_oe'] = int('о' in src and 'ө' in cand)
    feats['restore_u_to_ue'] = int('у' in src and 'ү' in cand)
    feats['restore_u_to_uu'] = int('у' in src and 'ұ' in cand)
    feats['restore_k_to_q'] = int('к' in src and 'қ' in cand)
    feats['restore_g_to_gh'] = int('г' in src and 'ғ' in cand)
    feats['restore_n_to_ng'] = int('н' in src and 'ң' in cand)
    feats['restore_i_to_dotless'] = int('и' in src and 'і' in cand)

    for suf in KAZAKH_SUFFIXES:
        feats[f'src_suf_{suf}'] = int(src.endswith(suf))
        feats[f'cand_suf_{suf}'] = int(cand.endswith(suf))

    feats['fix_tyn_to_tyn_ng'] = int(src.endswith('тын') and cand.endswith('тың'))
    feats['fix_tin_to_tin_ng'] = int(src.endswith('тин') and cand.endswith('тің'))
    feats['fix_dyn_to_dyn_ng'] = int(src.endswith('дын') and cand.endswith('дың'))
    feats['fix_din_to_din_ng'] = int(src.endswith('дин') and cand.endswith('дің'))
    feats['fix_nyn_to_nyn_ng'] = int(src.endswith('нын') and cand.endswith('ның'))
    feats['fix_nin_to_nin_ng'] = int(src.endswith('нин') and cand.endswith('нің'))
    feats['fix_ga_to_gha'] = int(src.endswith('га') and cand.endswith('ға'))
    feats['fix_ka_to_qa'] = int(src.endswith('ка') and cand.endswith('қа'))

    feats['left_gen_right_poss'] = int(
        left.endswith(('ның', 'нің', 'дың', 'дің', 'тың', 'тің')) and
        cand.endswith(('ы', 'і', 'сы', 'сі', 'лары', 'лері'))
    )

    feats['cand_dat_right_motion'] = int(
        cand.endswith(('ға', 'ге', 'қа', 'ке')) and
        right in MOTION_WORDS
    )

    return feats

## MaxEnt reranker үшін оқыту деректерін құру

⚠️ **Түзету**: бастапқы `.py` экспортында `maxent_df` айнымалысы пайдаланылғанымен
ешбір cell-де нақты құрылмаған еді (мүмкін экспорт кезінде сол cell түсіп қалған).
Төмендегі cell осы олқылықты жабады — `build_maxent_rows(train_df, ...)` арқылы
`maxent_df`-ты нақты генерациялайды.

In [44]:
def build_maxent_rows(df, max_rows=300000, negatives_per_positive=6):
    rows = []

    for _, row in df.iterrows():
        noisy_tokens = row['noisy_sentence'].split()
        gold_tokens = row['correct_sentence'].split()

        if len(noisy_tokens) != len(gold_tokens):
            continue

        for i, (src, gold) in enumerate(zip(noisy_tokens, gold_tokens)):
            if src == gold:
                continue

            cands = generate_candidates(src, max_candidates=35, use_vocab=False)
            cands = list(dict.fromkeys([gold] + cands))

            negatives = [c for c in cands if c != gold]
            random.shuffle(negatives)
            selected = [gold] + negatives[:negatives_per_positive]

            left = noisy_tokens[i - 1] if i > 0 else ''
            right = noisy_tokens[i + 1] if i + 1 < len(noisy_tokens) else ''

            for cand in selected:
                rows.append({
                    'src': src,
                    'candidate': cand,
                    'gold': gold,
                    'left': left,
                    'right': right,
                    'label': int(cand == gold)
                })

                if len(rows) >= max_rows:
                    return pd.DataFrame(rows)

    return pd.DataFrame(rows)


maxent_df = build_maxent_rows(train_df, max_rows=300000, negatives_per_positive=6)
print('maxent_df shape:', maxent_df.shape)
maxent_df.head()

maxent_df shape: (300000, 6)


,src,candidate,gold,left,right,label
0,басшысынин,басшысынің,басшысынің,мемлекет,өкімімен,1
1,басшысынин,бәсшысынин,басшысынің,мемлекет,өкімімен,0
2,басшысынин,басшысыниін,басшысынің,мемлекет,өкімімен,0
3,басшысынин,басшысыныин,басшысынің,мемлекет,өкімімен,0
4,басшысынин,басшыысынин,басшысынің,мемлекет,өкімімен,0


In [45]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_dict = [
    maxent_features(r.src, r.candidate, r.left, r.right)
    for r in maxent_df.itertuples()
]
y = maxent_df['label'].values

vec = DictVectorizer(sparse=True)
X_vec = vec.fit_transform(X_dict)

maxent = LogisticRegression(
    max_iter=300,
    class_weight='balanced',
    solver='liblinear'
)

maxent.fit(X_vec, y)

print('Train accuracy:', accuracy_score(y, maxent.predict(X_vec)))

with open(DATA_DIR / 'maxent_vectorizer.pkl', 'wb') as f:
    pickle.dump(vec, f)

with open(DATA_DIR / 'maxent_model.pkl', 'wb') as f:
    pickle.dump(maxent, f)

with open(DATA_DIR / 'word_freq.pkl', 'wb') as f:
    pickle.dump(word_freq, f)

print('MaxEnt артефактілері сақталды')

Train accuracy: 0.99492
MaxEnt артефактілері сақталды


## Инференс каскады: BERT детектор → кандидат генерация → MaxEnt reranker

In [46]:
detector_model = bert_detector.eval()

def predict_ok_err(tokens, threshold=0.5):
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    device = next(bert_detector.parameters()).device
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = bert_detector(**inputs)

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)

    word_ids = encoding.word_ids()

    word_probs = []
    seen = set()

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)

        p = probs[0, i, 1].item()
        word_probs.append(p)

    labels = ["ERR" if p > threshold else "OK" for p in word_probs]
    return labels, word_probs


def rerank_candidate(src, cand_list, left='', right=''):
    feats = [maxent_features(src, c, left, right) for c in cand_list]
    X = vec.transform(feats)
    scores = maxent.predict_proba(X)[:, 1]
    best_idx = int(np.argmax(scores))
    return cand_list[best_idx], float(scores[best_idx]), list(zip(cand_list, scores))


def correct_sentence(sentence, threshold=0.50, force_all=False):
    tokens = sentence.split()
    if not tokens:
        return sentence, []
    labels, probs = predict_ok_err(tokens, threshold=threshold)

    output = []
    debug = []

    for i, tok in enumerate(tokens):
        is_err = force_all or (i < len(labels) and labels[i] == 'ERR')

        if not is_err:
            output.append(tok)
            debug.append((tok, 'OK', tok, probs[i] if i < len(probs) else None, []))
            continue

        cands = generate_candidates(tok, max_candidates=30, use_vocab=False)

        left = tokens[i - 1] if i > 0 else ''
        right = tokens[i + 1] if i + 1 < len(tokens) else ''

        best, score, ranked = rerank_candidate(tok, cands, left, right)
        output.append(best)

        debug.append((
            tok, 'ERR', best,
            probs[i] if i < len(probs) else None,
            sorted(ranked, key=lambda x: x[1], reverse=True)[:5]
        ))

    return ' '.join(output), debug


example = 'мектептын окушылары сабака келди'
corrected, dbg = correct_sentence(example, threshold=0.30, force_all=True)

print('INPUT :', example)
print('OUTPUT:', corrected)
for item in dbg:
    print(item)


# ---------------------------------------------------------------------------
# Тыныс белгісін сақтайтын қаптама (нақты Whisper мәтіні үшін)
# ---------------------------------------------------------------------------
# Оқыту корпусында тыныс белгілері БӨЛЕК токен ретінде алынып тасталды
# (жоғарыдағы clean_kazakh_sentence-ті қараңыз), ал нақты Whisper мәтінінде
# тыныс белгісі сөзге ЖАБЫСЫП тұрады ("кезде," "ба?"). Сондықтан әр сөздің
# тыныс белгісін алдымен бөліп аламыз, "өзек" сөзге ғана түзету қолданамыз,
# содан кейін тыныс белгісін қайта жабыстырамыз — осылай VOCAB/BERT дұрыс
# "таза" сөзбен салыстырады.
_PUNCT_RE = re.compile(r"^(\W*)(.*?)(\W*)$", re.UNICODE)

def _split_punct(token):
    m = _PUNCT_RE.match(token)
    if not m or not m.group(2):
        return "", token, ""
    return m.group(1), m.group(2), m.group(3)

def correct_sentence_real(sentence, threshold=0.30, force_all=False):
    raw_tokens = sentence.split()
    if not raw_tokens:
        return sentence, []

    pre, cores, post = [], [], []
    for t in raw_tokens:
        p1, core, p2 = _split_punct(t)
        pre.append(p1)
        cores.append(core)
        post.append(p2)

    core_sentence = ' '.join(cores)
    corrected_core, debug = correct_sentence(core_sentence, threshold=threshold, force_all=force_all)
    corrected_tokens = corrected_core.split()

    # Ұзындық сирек жағдайда сай келмеуі мүмкін (бос "өзек" токендер салдарынан) —
    # сақтық үшін минимумын аламыз, қалғанын өзгеріссіз қалдырамыз.
    out = []
    n = min(len(corrected_tokens), len(pre), len(post))
    for i in range(n):
        out.append(pre[i] + corrected_tokens[i] + post[i])
    out.extend(corrected_tokens[n:])
    return ' '.join(out), debug


example_real = 'Тіп-тип келгенгезде, бәрі мәдінеттен басталаттағы да.'
corrected_real, _ = correct_sentence_real(example_real, threshold=0.30, force_all=True)
print('\nREAL INPUT :', example_real)
print('REAL OUTPUT:', corrected_real)


# ---------------------------------------------------------------------------
# Ұзын мәтіндер үшін chunk-теу (толық 100-аудио транскрипттер 250–1400 сөзге
# дейін жетеді, ал BERT tokenizer-дің максимумы ~512 subtoken — бір бүтін
# транскриптті "correct_sentence"-ке тұтас беру тыныш ҚИЫЛЫП (truncate)
# қалдырады да, соңғы сөздер ешқашан тексерілмейді. Сондықтан алдымен
# сөйлемдерге, содан кейін қажет болса сөз-терезелеріне бөліп өңдейміз.)
# ---------------------------------------------------------------------------
def _sentence_split(text):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]

def correct_long_text_real(text, max_words_per_chunk=80, threshold=0.30, force_all=False):
    sentences = _sentence_split(text)
    chunks, cur, cur_len = [], [], 0

    for s in sentences:
        wlen = len(s.split())
        if wlen > max_words_per_chunk:
            words = s.split()
            for i in range(0, len(words), max_words_per_chunk):
                chunks.append(' '.join(words[i:i + max_words_per_chunk]))
            continue
        if cur_len + wlen > max_words_per_chunk and cur:
            chunks.append(' '.join(cur))
            cur, cur_len = [], 0
        cur.append(s)
        cur_len += wlen

    if cur:
        chunks.append(' '.join(cur))

    out_chunks = []
    for ch in chunks:
        try:
            fixed, _ = correct_sentence_real(ch, threshold=threshold, force_all=force_all)
        except Exception:
            fixed = ch
        out_chunks.append(fixed)

    return ' '.join(out_chunks)

INPUT : мектептын окушылары сабака келди
OUTPUT: мектептың оқушылары сабақа келді
('мектептын', 'ERR', 'мектептың', 0.9798039197921753, [('мектептың', np.float64(0.9999111893800033)), ('меқтептын', np.float64(0.00037812693237363826)), ('мектаептын', np.float64(6.251925444165611e-05)), ('мектеіптын', np.float64(6.251925444165611e-05)), ('мектепетын', np.float64(6.251925444165611e-05))])
('окушылары', 'ERR', 'оқушылары', 0.9947078227996826, [('оқушылары', np.float64(0.9997644231279215)), ('океушылары', np.float64(6.251925444165611e-05)), ('оікушылары', np.float64(6.251925444165611e-05)), ('окушылареы', np.float64(6.251925444165611e-05)), ('окуішылары', np.float64(6.251925444165611e-05))])
('сабака', 'ERR', 'сабақа', 0.061092525720596313, [('сабақа', np.float64(0.0009143949123736986)), ('сабакеа', np.float64(6.251925444165611e-05)), ('сабеака', np.float64(6.251925444165611e-05)), ('саібака', np.float64(6.251925444165611e-05)), ('сабіака', np.float64(6.251925444165611e-05))])
('келди', 'ER

## Синтетикалық тест жиынында сапаны тексеру (sanity check)

Бұл — нағыз аудио емес, синтетикалық noisy/correct жұптары бойынша тексеру.
Каскадтың өзі дұрыс жұмыс істеп тұрғанын растайды, бірақ нағыз Whisper
шығысындағы қателер синтетикалыққа қарағанда әлдеқайда ауыр болуы мүмкін
(1-сатыдағы `recoverability_report` осыны көрсеткен болатын) — сондықтан
төмендегі синтетикалық сандарды нақты аудио нәтижелерімен ШАТАСТЫРМАҢЫЗ.

In [47]:
import jiwer

SAMPLE_EVAL = min(1000, len(test_df))

pred_texts, gold_texts, noisy_texts = [], [], []

for _, row in test_df.sample(SAMPLE_EVAL, random_state=SEED).iterrows():
    pred, _ = correct_sentence(row['noisy_sentence'], threshold=0.30, force_all=False)
    pred_texts.append(pred)
    gold_texts.append(row['correct_sentence'])
    noisy_texts.append(row['noisy_sentence'])

print('(Синтетикалық) Noisy WER:', jiwer.wer(gold_texts, noisy_texts))
print('(Синтетикалық) Post  WER:', jiwer.wer(gold_texts, pred_texts))
print('(Синтетикалық) Noisy CER:', jiwer.cer(gold_texts, noisy_texts))
print('(Синтетикалық) Post  CER:', jiwer.cer(gold_texts, pred_texts))

for i in range(min(5, SAMPLE_EVAL)):
    print('NOISY:', noisy_texts[i])
    print('PRED :', pred_texts[i])
    print('GOLD :', gold_texts[i])
    print()

(Синтетикалық) Noisy WER: 0.1364276529300104
(Синтетикалық) Post  WER: 0.029391953952605476
(Синтетикалық) Noisy CER: 0.019782033364060782
(Синтетикалық) Post  CER: 0.005820868892919801
NOISY: бұдан баске болгарлық депутаттар үшін қазақстан-2050 стратегиясынин ережелері жанепрезиденттын рухани жаңғыру мақаласы салмақты талдау тақырыбы болып табылады олар казақстаннің дүние жүзіндегі ең дамыген елдер қатарына кірудың келелі мақсатарына жетуі ушін нақты іс-қимыл бағдарламалары болып қабылданады
PRED : бұдан баске болгарлық депутаттар үшін қазақстан-2050 стратегиясынің ережелері жанепрезиденттың рухани жаңғыру мақаласы салмақты талдау тақырыбы болып табылады олар қазақстаннің дүние жүзіндегі ең дамыген елдер қатарына кірудың келелі мақсатарыңа жетуі үшін нақты іс-қимыл бағдарламалары болып қабылданады
GOLD : бұдан баске болгарлық депутаттар үшін қазақстан-2050 стратегиясынің ережелері жәнепрезиденттың рухани жаңғыру мақаласы салмақты талдау тақырыбы болып табылады олар қазақстаннің дүние 

## Нақты 100 аудио бойынша қолдану және 1-сатымен салыстыру

Бұл cell 1-сатыда сақталған `mapp_v2_batch_results.csv`-ті (Drive-тағы
`dataset_wav16k` қалтасында) оқиды — онда әр аудио үшін REF, RAW (Whisper
шикі), FIXED (лексикалық MAPP v2) мәтіндер бар. Whisper-ді ҚАЙТА ІСКЕ
ҚОСПАЙМЫЗ (қымбат/баяу) — тек RAW мәтінге BERT+MaxEnt каскадын қолданамыз
да, сол RAW/лексикалық-FIXED-пен бірдей 6 метриканы есептейміз.

In [48]:
STAGE1_CSV = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_v2_batch_results_largev3_lex.csv"

if not os.path.exists(STAGE1_CSV):
    print(
        f"[ЕСКЕРТУ] {STAGE1_CSV} табылмады.\n"
        "Бұл bigcorpus RAW негізіндегі ref/raw/fixed CSV-і — алдымен "
        "mapp_v2_batch_results_bigcorpus.csv файлын дәл осы жолға "
        "(Drive-тағы bigcorpus/ қалтасына) жүктеп қойыңыз, содан кейін "
        "бұл cell-ді қайта іске қосыңыз."
    )
    stage1_df = None
else:
    stage1_df = pd.read_csv(STAGE1_CSV)
    print("kk-turbo+VAD+decloop RAW нәтижесі жүктелді:", stage1_df.shape)
    print("Өңделген аудио саны:", len(stage1_df))
    display(stage1_df[["duration", "stem", "ref", "raw", "fixed"]].head())


kk-turbo+VAD+decloop RAW нәтижесі жүктелді: (100, 5)
Өңделген аудио саны: 100


,duration,stem,ref,raw,fixed
0,30s,30s_001,"Түптеп келген кезде, бәрі мәдениеттен басталад...",Түптеп келген кезде бәрі мәдениеттен басталатт...,Түптеп келген кезде бәрі мәдениеттен басталаст...
1,30s,30s_002,Мемлекеттің бекітіп берген стандарты бар. Сол ...,Мемлекеттің белгіп берген стандарты бар. Станд...,Мемлекеттің белгіі берген стандарты бар. Станд...
2,30s,30s_003,Бұл екі мың он сегізінші жылдардан басталған б...,"Бұл 2018 жылдардың басталан болатын, сол заман...","Бұл 2018 жылдардың бастала болатын, сол заманд..."
3,30s,30s_004,Сіз негізі осы қаржы және несие мамандығы бойы...,Сіз негізі осы қаржы және несие мамадыға бойын...,Сіз негізі осы қаржы және несие мамандыға бойы...
4,30s,30s_005,Қаржылық сауаттылық пен жеке қаржы туралы білі...,Қаржылық сауатылық пен жеке қаржы туралы білім...,Қаржылық сауаттылық пен жеке қаржы туралы білі...


In [49]:
!pip -q install sacrebleu bert_score

In [50]:
import re as _re_metrics
import sacrebleu
from bert_score import score as bertscore_fn

def norm_text(t):
    # Тыныс белгісі мен әріп регистрі WER/CER-ді жасанды түрде көтермеу
    # үшін салыстырудан бұрын мәтінді нормализациялаймыз (1-саты
    # notebook-тағы норм функциясымен бірдей).
    t = str(t).lower()
    t = _re_metrics.sub(r"[^\w\s]", "", t)
    t = _re_metrics.sub(r"\s+", " ", t).strip()
    return t

def wer_m(ref, hyp):
    return jiwer.wer(norm_text(ref), norm_text(hyp))

def cer_m(ref, hyp):
    return jiwer.cer(norm_text(ref), norm_text(hyp))

def bleu_m(ref, hyp):
    return sacrebleu.sentence_bleu(norm_text(hyp), [norm_text(ref)]).score

def chrf_m(ref, hyp):
    return sacrebleu.sentence_chrf(norm_text(hyp), [norm_text(ref)]).score

def ter_m(ref, hyp):
    return sacrebleu.metrics.TER().sentence_score(norm_text(hyp), [norm_text(ref)]).score

def bertscore_m(ref, hyp, lang="multilingual"):
    # BERTScore семантикалық ұқсастықты өлшейді, пунктуацияға әлдеқайда аз
    # сезімтал, бірақ бірізділік үшін мұнда да нормализацияланған мәтінді
    # қолданамыз.
    P, R, F1 = bertscore_fn([norm_text(hyp)], [norm_text(ref)], lang="other", model_type="bert-base-multilingual-cased")
    return float(F1[0])

In [51]:
from tqdm.auto import tqdm

if stage1_df is not None:
    bert_rows = []
    for _, r in tqdm(stage1_df.iterrows(), total=len(stage1_df), desc="BERT+MaxEnt cascade"):
        ref_text = str(r["ref"])
        raw_text = str(r["raw"])
        try:
            bert_fixed_text = correct_long_text_real(raw_text, max_words_per_chunk=80, threshold=0.30, force_all=False)
        except Exception as e:
            print(f"[ERROR] {r['duration']}/{r['stem']}: {e}")
            bert_fixed_text = raw_text

        bert_rows.append({
            "duration": r["duration"],
            "stem": r["stem"],
            "ref": ref_text,
            "raw": raw_text,
            "lexical_fixed": r["fixed"],
            "bert_fixed": bert_fixed_text,
            "wer_raw": wer_m(ref_text, raw_text),
            "wer_lexical": wer_m(ref_text, r["fixed"]),
            "wer_bert": wer_m(ref_text, bert_fixed_text),
            "cer_raw": cer_m(ref_text, raw_text),
            "cer_lexical": cer_m(ref_text, r["fixed"]),
            "cer_bert": cer_m(ref_text, bert_fixed_text),
            "bleu_raw": bleu_m(ref_text, raw_text),
            "bleu_lexical": bleu_m(ref_text, r["fixed"]),
            "bleu_bert": bleu_m(ref_text, bert_fixed_text),
            "chrf_raw": chrf_m(ref_text, raw_text),
            "chrf_lexical": chrf_m(ref_text, r["fixed"]),
            "chrf_bert": chrf_m(ref_text, bert_fixed_text),
            "ter_raw": ter_m(ref_text, raw_text),
            "ter_lexical": ter_m(ref_text, r["fixed"]),
            "ter_bert": ter_m(ref_text, bert_fixed_text),
            "bertscore_raw": bertscore_m(ref_text, raw_text),
            "bertscore_lexical": bertscore_m(ref_text, r["fixed"]),
            "bertscore_bert": bertscore_m(ref_text, bert_fixed_text),
        })

    stage2_df = pd.DataFrame(bert_rows)
    OUT_CSV = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage2_bert_maxent_results_largev3_bigcorpus.csv")
    stage2_df.to_csv(OUT_CSV, index=False)
    print("Сақталды:", OUT_CSV)
    stage2_df.head()
else:
    stage2_df = None

BERT+MaxEnt cascade:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_stage2_bert_maxent_results_largev3_bigcorpus.csv


### Қорытынды салыстыру: RAW vs Лексикалық MAPP v2 (1-саты) vs BERT+MaxEnt (2-саты)

In [52]:
if stage2_df is not None:
    # Excel (.xlsx) кейбір control-таңбаларды қабылдамайды — Whisper/каскад
    # мәтіндерінде кездесуі мүмкін, сондықтан жазбас бұрын тазалаймыз.
    import re as _re
    _ILLEGAL_XLSX_RE = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

    def _clean_for_excel(val):
        if isinstance(val, str):
            return _ILLEGAL_XLSX_RE.sub("", val)
        return val

    stage2_df = stage2_df.applymap(_clean_for_excel)

    metric_groups = ["wer", "cer", "bleu", "chrf", "ter", "bertscore"]
    summary_rows = []
    for m in metric_groups:
        summary_rows.append({
            "metric": m,
            "raw": stage2_df[f"{m}_raw"].mean(),
            "lexical_mapp_v2 (1-саты, largev3 негізінде)": stage2_df[f"{m}_lexical"].mean(),
            "bert_maxent (2-саты, largev3 негізінде)": stage2_df[f"{m}_bert"].mean(),
        })
    comparison_df = pd.DataFrame(summary_rows).round(4)
    display(comparison_df)

    OUT_XLSX = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage2_on_largev3_comparison_bigcorpus.xlsx")
    with pd.ExcelWriter(OUT_XLSX) as writer:
        stage2_df.to_excel(writer, sheet_name="per_file", index=False)
        comparison_df.to_excel(writer, sheet_name="summary_comparison", index=False)
        stage2_df.groupby("duration")[[f"{m}_raw" for m in metric_groups] +
                                        [f"{m}_lexical" for m in metric_groups] +
                                        [f"{m}_bert" for m in metric_groups]].mean().round(4).to_excel(
            writer, sheet_name="by_duration"
        )
    print("Excel сақталды:", OUT_XLSX)
else:
    print("stage2_df жоқ — жоғарыдағы 1-саты CSV-ін жүктеу cell-ін қараңыз.")

/tmp/ipykernel_1696/1001861953.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  stage2_df = stage2_df.applymap(_clean_for_excel)


,metric,raw,"lexical_mapp_v2 (1-саты, largev3 негізінде)","bert_maxent (2-саты, largev3 негізінде)"
0,wer,0.6824,0.6953,0.6835
1,cer,0.3987,0.4105,0.4026
2,bleu,16.6371,15.6348,16.6357
3,chrf,57.1489,55.2446,56.6429
4,ter,73.3851,74.5040,73.4454
5,bertscore,0.7821,0.7739,0.7801


Excel сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_stage2_on_largev3_comparison_bigcorpus.xlsx


## 3-саты: MaxEnt-ті СИНТЕТИКАЛЫҚ емес, НАҚТЫ 100 аудионың raw↔ref жұптарында қайта оқыту

**Неге керек:** жоғарыдағы BERT-детектор мен MaxEnt reranker таза қазақ
корпусынан жасанды бұрмаланған (`CHAR_RULES`/`SUFFIX_RULES`) мәтінде
оқытылған — нақты Whisper large-v3 қателерін ешқашан көрген жоқ
(train/test distribution mismatch, бұрынғы root-cause талдауда табылған
басты проблема). Бұл cell-дер сол олқылықты жабады:

1. `stage1_df`-тегі **нақты `raw`/`ref` жұбын** сөз деңгейінде туралайды
   (`difflib`, тек ұзындығы бірдей replace-блоктар — қауіпсіз).
2. Осы нақты (src→gold) жұптардан MaxEnt reranker-ді **қайта оқытады**,
   морфологиялық белгілер (`vowel_harmony_ok`, `restore_*`,
   `left_gen_right_poss`, `cand_dat_right_motion`, суффикс-белгілер)
   сол күйі қалады — бұлар `maxent_features()`-те бұрыннан бар.
3. **Деректің аздығынан (100 жазба) ағып кетуді (leakage) болдырмау үшін
   5-fold cross-validation** қолданылады: әр fold өз үлесін көрмеген
   моделден алады (дәл `crf_code/bertgate.py`-дегі әдіспен бірдей).
4. BERT қате-детекторы ӨЗГЕРМЕЙДІ (синтетикада оқытылған күйі қалады) —
   тек reranker (MaxEnt) бөлігі нақты деректен үйренеді.

Нәтиже — жаңа баған `real_morph_fixed`, оны RAW / 1-саты (lexical) /
2-саты (bert_fixed, синтетика) үшеуімен бірге салыстырамыз.

In [53]:
import difflib

def _core_tokens(text):
    """Тыныс белгісін бөліп алып, кіші әріпке келтірілген "өзек" сөздер.
    VOCAB/generate_candidates дәл осындай таза токенмен жұмыс істейді."""
    toks = str(text).split()
    cores = []
    for t in toks:
        _, core, _ = _split_punct(t)
        cores.append(core.lower())
    return [c for c in cores if c]


def align_words(raw_tokens, ref_tokens):
    """Тек ҰЗЫНДЫҒЫ бірдей equal/replace блоктарын аламыз -- сатылас
    сөзбе-сөз сәйкестік болмаса гold белгісіз болып қалады, ол дұрыс
    (кірістіру/жойылу орнына жалған src->gold жұп жасамаймыз)."""
    sm = difflib.SequenceMatcher(None, raw_tokens, ref_tokens, autojunk=False)
    out = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag in ("equal", "replace") and (i2 - i1) == (j2 - j1):
            for k in range(i2 - i1):
                out.append((raw_tokens[i1 + k], ref_tokens[j1 + k], i1 + k))
    return out


def build_real_maxent_rows(df, negatives_per_positive=6, max_candidates=35, seed=SEED):
    rows = []
    rnd = random.Random(seed)
    for _, r in df.iterrows():
        raw_tokens = _core_tokens(r["raw"])
        ref_tokens = _core_tokens(r["ref"])
        for src, gold, pos in align_words(raw_tokens, ref_tokens):
            if src == gold:
                continue
            cands = generate_candidates(src, max_candidates=max_candidates, use_vocab=False)
            cands = list(dict.fromkeys([gold] + cands))
            negatives = [c for c in cands if c != gold]
            rnd.shuffle(negatives)
            selected = [gold] + negatives[:negatives_per_positive]
            left = raw_tokens[pos - 1] if pos > 0 else ""
            right = raw_tokens[pos + 1] if pos + 1 < len(raw_tokens) else ""
            for cand in selected:
                rows.append({"src": src, "candidate": cand, "gold": gold,
                             "left": left, "right": right, "label": int(cand == gold)})
    return pd.DataFrame(rows)


print("Сынақ: бірінші жазбадағы туралау")
_ex = stage1_df.iloc[0]
print(align_words(_core_tokens(_ex["raw"]), _core_tokens(_ex["ref"]))[:10])

Сынақ: бірінші жазбадағы туралау
[('түптеп', 'түптеп', 0), ('келген', 'келген', 1), ('кезде', 'кезде', 2), ('бәрі', 'бәрі', 3), ('мәдениеттен', 'мәдениеттен', 4), ('басталаттағы', 'басталады', 5), ('да', 'тағыда', 6), ('тамақтану', 'тамақтану', 7), ('мәдениеті', 'мәдениеті', 8), ('чекап', 'чек-ап', 9)]


In [54]:
N_FOLDS = 5
row_ids = list(range(len(stage1_df)))
random.Random(SEED).shuffle(row_ids)
folds = [row_ids[i::N_FOLDS] for i in range(N_FOLDS)]

fold_models = {}   # test-fold index -> (vec_real, maxent_real)

for f_idx, test_ids in enumerate(folds):
    test_set = set(test_ids)
    train_ids = [i for i in row_ids if i not in test_set]
    train_part = stage1_df.iloc[train_ids]

    real_df = build_real_maxent_rows(train_part, negatives_per_positive=6, max_candidates=35)

    X_dict = [maxent_features(r.src, r.candidate, r.left, r.right) for r in real_df.itertuples()]
    y = real_df["label"].values

    vec_real = DictVectorizer(sparse=True)
    X_vec = vec_real.fit_transform(X_dict)

    maxent_real = LogisticRegression(max_iter=300, class_weight="balanced", solver="liblinear")
    maxent_real.fit(X_vec, y)

    fold_models[f_idx] = (vec_real, maxent_real)
    print(f"fold {f_idx}: train жұп саны {len(real_df)}, train acc "
          f"{accuracy_score(y, maxent_real.predict(X_vec)):.3f}")

row_to_fold = {}
for f_idx, ids in enumerate(folds):
    for i in ids:
        row_to_fold[i] = f_idx

with open(DATA_DIR / "real_morph_maxent_folds.pkl", "wb") as f:
    pickle.dump({"fold_models": fold_models, "row_to_fold": row_to_fold}, f)
print("5-fold MaxEnt (нақты морфология) дайын және сақталды.")

fold 0: train жұп саны 26373, train acc 0.981
fold 1: train жұп саны 24838, train acc 0.980
fold 2: train жұп саны 25803, train acc 0.981
fold 3: train жұп саны 21680, train acc 0.983
fold 4: train жұп саны 25314, train acc 0.982
5-fold MaxEnt (нақты морфология) дайын және сақталды.


In [55]:
def rerank_candidate_with(src, cand_list, left, right, vec_m, maxent_m):
    feats = [maxent_features(src, c, left, right) for c in cand_list]
    X = vec_m.transform(feats)
    scores = maxent_m.predict_proba(X)[:, 1]
    best_idx = int(np.argmax(scores))
    return cand_list[best_idx]


def correct_sentence_with(sentence, vec_m, maxent_m, threshold=0.30, force_all=False):
    tokens = sentence.split()
    if not tokens:
        return sentence
    labels, probs = predict_ok_err(tokens, threshold=threshold)
    output = []
    for i, tok in enumerate(tokens):
        is_err = force_all or (i < len(labels) and labels[i] == "ERR")
        if not is_err:
            output.append(tok)
            continue
        cands = generate_candidates(tok, max_candidates=30, use_vocab=False)
        left = tokens[i - 1] if i > 0 else ""
        right = tokens[i + 1] if i + 1 < len(tokens) else ""
        output.append(rerank_candidate_with(tok, cands, left, right, vec_m, maxent_m))
    return " ".join(output)


def correct_sentence_real_with(sentence, vec_m, maxent_m, threshold=0.30, force_all=False):
    raw_tokens = sentence.split()
    if not raw_tokens:
        return sentence
    pre, cores, post = [], [], []
    for t in raw_tokens:
        p1, core, p2 = _split_punct(t)
        pre.append(p1); cores.append(core); post.append(p2)
    corrected_core = correct_sentence_with(" ".join(cores), vec_m, maxent_m, threshold, force_all)
    corrected_tokens = corrected_core.split()
    out, n = [], min(len(corrected_tokens), len(pre), len(post))
    for i in range(n):
        out.append(pre[i] + corrected_tokens[i] + post[i])
    out.extend(corrected_tokens[n:])
    return " ".join(out)


def correct_long_text_with(text, vec_m, maxent_m, max_words_per_chunk=80, threshold=0.30, force_all=False):
    sentences = _sentence_split(text)
    chunks, cur, cur_len = [], [], 0
    for s in sentences:
        wlen = len(s.split())
        if wlen > max_words_per_chunk:
            words = s.split()
            for i in range(0, len(words), max_words_per_chunk):
                chunks.append(" ".join(words[i:i + max_words_per_chunk]))
            continue
        if cur_len + wlen > max_words_per_chunk and cur:
            chunks.append(" ".join(cur)); cur, cur_len = [], 0
        cur.append(s); cur_len += wlen
    if cur:
        chunks.append(" ".join(cur))
    out_chunks = []
    for ch in chunks:
        try:
            out_chunks.append(correct_sentence_real_with(ch, vec_m, maxent_m, threshold, force_all))
        except Exception:
            out_chunks.append(ch)
    return " ".join(out_chunks)

print("Fold-бойынша cascade функциялары дайын.")

Fold-бойынша cascade функциялары дайын.


In [56]:
real_rows = []
for i, (_, r) in enumerate(tqdm(stage1_df.iterrows(), total=len(stage1_df), desc="Real-morph MaxEnt cascade (5-fold)")):
    f_idx = row_to_fold[i]
    vec_m, maxent_m = fold_models[f_idx]
    ref_text = str(r["ref"]); raw_text = str(r["raw"])
    try:
        real_fixed = correct_long_text_with(raw_text, vec_m, maxent_m, max_words_per_chunk=80,
                                            threshold=0.30, force_all=False)
    except Exception as e:
        print(f"[ERROR] {r['duration']}/{r['stem']}: {e}")
        real_fixed = raw_text

    real_rows.append({
        "duration": r["duration"], "stem": r["stem"], "fold": f_idx,
        "ref": ref_text, "raw": raw_text,
        "lexical_fixed": r["fixed"],
        "bert_fixed_synthetic": stage2_df.loc[stage2_df["stem"] == r["stem"], "bert_fixed"].values[0]
            if stage2_df is not None and (stage2_df["stem"] == r["stem"]).any() else None,
        "real_morph_fixed": real_fixed,
        "wer_raw": wer_m(ref_text, raw_text),
        "wer_lexical": wer_m(ref_text, r["fixed"]),
        "wer_real_morph": wer_m(ref_text, real_fixed),
        "cer_raw": cer_m(ref_text, raw_text),
        "cer_real_morph": cer_m(ref_text, real_fixed),
        "bleu_raw": bleu_m(ref_text, raw_text),
        "bleu_real_morph": bleu_m(ref_text, real_fixed),
        "chrf_raw": chrf_m(ref_text, raw_text),
        "chrf_real_morph": chrf_m(ref_text, real_fixed),
        "ter_raw": ter_m(ref_text, raw_text),
        "ter_real_morph": ter_m(ref_text, real_fixed),
        "bertscore_raw": bertscore_m(ref_text, raw_text),
        "bertscore_real_morph": bertscore_m(ref_text, real_fixed),
    })

real_morph_df = pd.DataFrame(real_rows)
OUT_CSV3 = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage3_real_morph_maxent_results.csv")
real_morph_df.to_csv(OUT_CSV3, index=False)
print("Сақталды:", OUT_CSV3)
real_morph_df.head()

Real-morph MaxEnt cascade (5-fold):   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_stage3_real_morph_maxent_results.csv


,duration,stem,fold,ref,raw,lexical_fixed,bert_fixed_synthetic,real_morph_fixed,wer_raw,wer_lexical,...,cer_raw,cer_real_morph,bleu_raw,bleu_real_morph,chrf_raw,chrf_real_morph,ter_raw,ter_real_morph,bertscore_raw,bertscore_real_morph
0,30s,30s_001,0,"Түптеп келген кезде, бәрі мәдениеттен басталад...",Түптеп келген кезде бәрі мәдениеттен басталатт...,Түптеп келген кезде бәрі мәдениеттен басталаст...,Түптеп келген кезде бәрі мәдениеттен басталатт...,Түптеп келген кезде бәрі мәдениеттен басталатт...,0.342466,0.410959,...,0.085239,0.093555,40.948485,36.070608,85.539179,83.237284,34.246575,38.356164,0.914568,0.900070
1,30s,30s_002,1,Мемлекеттің бекітіп берген стандарты бар. Сол ...,Мемлекеттің белгіп берген стандарты бар. Станд...,Мемлекеттің белгіі берген стандарты бар. Станд...,Мемлекеттің белгіп берген стандарты бар. Стаңд...,Мемлекеттің белгіп берген стандарты бар. Стаңд...,0.488889,0.511111,...,0.158774,0.164345,21.730681,21.730681,75.910898,74.110453,48.888889,48.888889,0.861000,0.904332
2,30s,30s_003,2,Бұл екі мың он сегізінші жылдардан басталған б...,"Бұл 2018 жылдардың басталан болатын, сол заман...","Бұл 2018 жылдардың бастала болатын, сол заманд...","Бұл 2018 жылдардың басталан болатын, сол заман...","Бұл 2018 жылдардың басталан болатын, сол заман...",0.533333,0.533333,...,0.185714,0.197619,22.213396,22.213396,67.743489,66.002837,53.333333,53.333333,0.845688,0.833943
3,30s,30s_004,2,Сіз негізі осы қаржы және несие мамандығы бойы...,Сіз негізі осы қаржы және несие мамадыға бойын...,Сіз негізі осы қаржы және несие мамандыға бойы...,Сіз негізі осы қаржы және несие мамадыға бойын...,Сіз негізі осы қаржы және несие мамадыға бойын...,0.450980,0.470588,...,0.191045,0.194030,32.050192,32.050192,70.598432,70.237277,45.098039,45.098039,0.869621,0.864164
4,30s,30s_005,0,Қаржылық сауаттылық пен жеке қаржы туралы білі...,Қаржылық сауатылық пен жеке қаржы туралы білім...,Қаржылық сауаттылық пен жеке қаржы туралы білі...,Қаржылық сауатылық пен жеке қаржы туралы білім...,Қаржылық сауатылық пен жеке қаржы туралы білім...,1.327869,1.344262,...,0.930456,0.932854,20.138748,17.281866,65.760750,64.747443,132.786885,134.426230,0.808571,0.803681


In [57]:
if stage2_df is not None:
    merged3 = real_morph_df.merge(
        stage2_df[["stem", "wer_bert", "cer_bert", "bleu_bert", "chrf_bert", "ter_bert", "bertscore_bert"]],
        on="stem", how="left"
    )
    rows3 = []
    for m in ["wer", "cer", "bleu", "chrf", "ter", "bertscore"]:
        rows3.append({
            "metric": m,
            "raw": merged3[f"{m}_raw"].mean(),
            "1-саты lexical_mapp_v2": merged3[f"{m}_lexical"].mean() if f"{m}_lexical" in merged3 else None,
            "2-саты bert_maxent (синтетика)": merged3[f"{m}_bert"].mean(),
            "3-саты bert_maxent+морф (НАҚТЫ 100 аудио, 5-fold CV)": merged3[f"{m}_real_morph"].mean(),
        })
    final_compare_df = pd.DataFrame(rows3).round(4)
    display(final_compare_df)

    OUT_XLSX3 = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage3_real_morph_comparison.xlsx")
    with pd.ExcelWriter(OUT_XLSX3) as writer:
        merged3.to_excel(writer, sheet_name="per_file", index=False)
        final_compare_df.to_excel(writer, sheet_name="summary_comparison", index=False)
    print("Excel сақталды:", OUT_XLSX3)
else:
    print("stage2_df жоқ -- 2-саты (синтетика) бағанасыз салыстырамыз, тек RAW/1-саты/3-саты.")

,metric,raw,1-саты lexical_mapp_v2,2-саты bert_maxent (синтетика),"3-саты bert_maxent+морф (НАҚТЫ 100 аудио, 5-fold CV)"
0,wer,0.6824,0.6953,0.6835,0.6837
1,cer,0.3987,NaN,0.4026,0.4025
2,bleu,16.6371,NaN,16.6357,16.5678
3,chrf,57.1489,NaN,56.6429,56.6937
4,ter,73.3851,NaN,73.4454,73.4572
5,bertscore,0.7821,NaN,0.7801,0.7815


Excel сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_stage3_real_morph_comparison.xlsx


## 4-саты: BERT + морфология + шынайы CRF (global normalization)

Бұрынғы 3-саты (`real_morph_fixed`) әр сөзді БӨЛЕК бағалайтын MaxEnt
(=MEMM) болатын. Бұл жерде `crf_code/src/crf.py`-дегі алгоритмді
(forward-backward, глобал нормировка, Viterbi) сіздің нақты 100
аудиоңызға бейімдеп порттаймыз. Үш бөлік:

- **BERT** -- бар детектордың қате-ықтималдығы (`bert_err`) emission
  белгісі ретінде қосылады (§13 құжаттың 1-варианты).
- **Морфология** -- transition белгілер ψ(prev,cur): GEN+POSS,
  DAT+MOTION, ACC+TRANS (§7C құжат), CSE сөздігі орнына жұрнақ/түбір
  эвристикасымен (толық CSE талдаушы қажет болса -- `qaz_stems`/
  `qaz_endings` файлдарын беріңіз, кейін ауыстырамыз).
- **CRF** -- бүкіл сөйлем біргелікте Viterbi-мен декодталады, оқыту
  forward-backward + L-BFGS (`crf_code/crf.py`-мен бірдей математика).

⚠️ **Адалдық үшін ескерту:** `mlm_logp`/`mlm_rank` (masked-LM негізді
BERT белгісі, §13-тің толық нұсқасы) бұл жерде ҚОСЫЛМАҒАН -- ол бөлек
GPU-сыйымды проход қажет етеді (`bertfeat.py`-дегідей). Қазір тек
`bert_err` (бар детектордан) қолданылады. Қаласаңыз, келесі қадамда
қосамыз.

### Жаңарту: эвристика орнына толық CSE морфологиялық талдаушы

Сіз жіберген `qaz_stems_unik_edit.xlsx` (103 616 түбір) мен
`qaz_ending_5994.xlsx` (5 994 жалғау, толық морф-талдаумен, мысалы
`<NB>*ға<dat>`) енді нақты септік/тәуелдік/есімше талдауын береді —
бұрынғы жұрнақ-эвристика (`GEN_SUF`/`POSS_SUF`/...) толықтай
ауыстырылады. Бұл — `crf_code/src/morphtrans.py`-дегі `Tagger`/
`TransFeats` класының сол алгоритмі (сингармониялық жұп, сөз табын
корпус бойынша дауыс беріп ажырату, GEN+POSS/DAT+MOTION/ACC+TRANS
белгілері), тек кеш файл орнына сіздің осы сеансыңыздағы Excel-дерден
тікелей құрылады.

In [59]:
# ---------------------------------------------------------------
# Биграмма ЯМ (алдыңғы cell-ден өзгеріссіз көшірілді -- ол cell endi
# CSE талдаушымен ауыстырылғандықтан осында қайта анықталады)
# ---------------------------------------------------------------
from collections import Counter as _Counter

_BI = _Counter()
_UNI_CTX = _Counter()
for _s in clean_sentences:
    _ws = ["<s>"] + _s.split() + ["</s>"]
    for _a, _b in zip(_ws, _ws[1:]):
        _BI[(_a, _b)] += 1
        _UNI_CTX[_a] += 1

_BI_K = 0.1
_BI_V = max(len(word_freq), 1)

def bigram_logp(prev, cur):
    num = _BI.get((prev, cur), 0) + _BI_K
    den = _UNI_CTX.get(prev, 0) + _BI_K * _BI_V
    return math.log(num / den)

MOTION_STEMS = {"бар", "кел", "кет", "шық", "кір", "қайт", "жет", "ұш", "жүр",
                "жүгір", "жүз", "көш", "аттан", "қон", "түс", "өт", "апар",
                "әкел", "әкет", "орал", "жылжы", "көтеріл", "жақында", "ауыс"}
TRANSITIVE_STEMS = {"көр", "ал", "бер", "жаса", "оқы", "жаз", "айт", "же", "іш",
                     "тап", "сал", "қой", "аш", "жап", "тыңда", "түсін", "біл",
                     "ұста", "сат", "кес", "жу", "таңда", "шеш", "көтер",
                     "тексер", "дайында", "аяқта", "баста", "бітір", "орында"}

# ---------------------------------------------------------------
# Толық CSE морфологиялық талдаушы (crf_code/src/morphtrans.py портты)
# ---------------------------------------------------------------
import re as _re_morph
from collections import defaultdict as _dd

STEMS_XLSX = "/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/qaz_stems_unik_edit.xlsx"
ENDINGS_XLSX = "/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/qaz_ending_5994.xls"
# ^ жоғарыдағы екі файлды Drive-тағы осы қалтаға (немесе өз жолыңызға) салыңыз

_HOMO = {"i": "і", "a": "а", "e": "е", "o": "о", "p": "р", "c": "с",
         "x": "х", "y": "у", "k": "к", "m": "м", "t": "т", "ј": "і", "h": "һ"}
_VPAIR = {"а": "е", "е": "а", "о": "ө", "ө": "о", "ұ": "ү", "ү": "ұ", "ы": "і", "і": "ы"}

def _mclean(s):
    return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())

def _mtwin(e):
    return "".join(_VPAIR.get(c, c) for c in e)

CASES = ["nom", "gen", "dat", "acc", "loc", "abl", "ins"]
CASE_TAG = {"gen": 1, "dat": 2, "acc": 3, "loc": 4, "abl": 5,
            "ınst": 6, "ınstr": 6, "instr": 6, "inst": 6}
FINITE = {"pst", "per", "tpat", "ıft", "tft", "aft", "apt", "tpt", "ppt",
          "mdes", "mimp", "mcond"}

MOTION = MOTION_STEMS       # жоғарыда §7C эвристикада анықталған тізім, қайта қолданамыз
TRANSITIVE = TRANSITIVE_STEMS


class Tagger:
    def __init__(self, stems, ana, votes):
        self.S = stems
        self.ana = ana
        self.votes = votes
        self.maxe = max((len(e) for e in ana), default=0)
        self._w = {}

    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}

        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str):
                continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags:
                continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k:
                    ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}

        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)

    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha():
                continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe:
                    break
                stem, end = w[:k], w[k:]
                if stem not in S:
                    continue
                if end in vb:
                    votes[stem][0] += n
                elif end in nb:
                    votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}

    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0))
        t = v + n
        if t < 5:
            return 2
        r = v / t
        if r > 0.6:
            return 1
        if r < 0.4:
            return 0
        return 2

    def word(self, w):
        t = self._w.get(w)
        if t is None:
            t = self._analyze(w)
            self._w[w] = t
        return t

    def _analyze(self, w):
        if not w or not w.isalpha():
            return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w)
            return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)

        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe:
                break
            stem, end = w[:k], w[k:]
            if stem not in self.S:
                continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem)
                sc = 0.1 * len(end)
                if "VB" in tags:
                    sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags:
                    sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc:
                    best, best_sc = (stem, tags, sp), sc
        if best is None:
            return (2, 0, False, False, False, False, False)

        stem, tags, sp = best
        if "VB" in tags and "NB" not in tags:
            pos = 1
        elif "NB" in tags and "VB" not in tags:
            pos = 0
        else:
            pos = sp
        case = 0
        for t in tags:
            if t in CASE_TAG:
                case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)


def _cse_names():
    n = ["tr_pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
    n += ["tr_%s_poss" % c for c in CASES]
    n += ["tr_%s_verb" % c for c in CASES]
    n += ["tr_dat_motion", "tr_acc_trans", "tr_nom_fin",
          "tr_same_case", "tr_both_bad", "tr_gen_nonposs"]
    return n

CSE_TR_NAMES = _cse_names()
N_CSE_TR = len(CSE_TR_NAMES)


class TransFeatsCSE:
    def __init__(self, tagger):
        self.tg = tagger
        self._pair = {}

    def pair(self, prev, cur):
        a, b = self.tg.word(prev), self.tg.word(cur)
        v = self._pair.get((a, b))
        if v is None:
            v = self._build(a, b)
            self._pair[(a, b)] = v
        return v

    @staticmethod
    def _build(a, b):
        pa, ca, _, _, _, _, oka = a
        pb, cb, possb, finb, motb, trb, okb = b
        f = np.zeros(N_CSE_TR)
        f[pa * 3 + pb] = 1.0
        o = 9
        if possb:
            f[o + ca] = 1.0
        o += len(CASES)
        if pb == 1:
            f[o + ca] = 1.0
        o += len(CASES)
        f[o + 0] = 1.0 if (ca == 2 and motb) else 0.0
        f[o + 1] = 1.0 if (ca == 3 and trb) else 0.0
        f[o + 2] = 1.0 if (pa == 0 and ca == 0 and finb) else 0.0
        f[o + 3] = 1.0 if (ca == cb and ca != 0) else 0.0
        f[o + 4] = 1.0 if (not oka and not okb) else 0.0
        f[o + 5] = 1.0 if (ca == 1 and not possb) else 0.0
        return f


_cse_tagger = Tagger.build(STEMS_XLSX, ENDINGS_XLSX, word_freq)
_trans_cse = TransFeatsCSE(_cse_tagger)

N_TR = 1 + N_CSE_TR  # [bigram_logp] + CSE pair-белгілер (GEN+POSS/DAT+MOTION/ACC+TRANS/т.б.)

def trans_feats(prev_c, cur_c):
    f = np.empty(N_TR)
    f[0] = bigram_logp(prev_c, cur_c) / 10.0
    f[1:] = _trans_cse.pair(prev_c, cur_c)
    return f

print("Толық CSE transition белгілер дайын, N_TR =", N_TR, "(1 биграмма + %d CSE)" % N_CSE_TR)

CSE: түбір 102743, жалғау-үлгі 6660
Толық CSE transition белгілер дайын, N_TR = 30 (1 биграмма + 29 CSE)


In [ ]:
# ---------------------------------------------------------------
# Runs құру: тізбектес позициялар, онда эталон candidate-тер ішінде бар
# (crf_code/src/crf.py-дегі build_runs логикасымен бірдей)
# ---------------------------------------------------------------
def build_runs_for_file(raw_text, ref_text, min_len=3, max_candidates=35):
    raw_tokens = _core_tokens(raw_text)
    ref_tokens = _core_tokens(ref_text)
    ref_map = {pos: g for s, g, pos in align_words(raw_tokens, ref_tokens) if s != g}
    # copy-позициялар да runs-та керек (s==g), эталон = өзі
    for s, g, pos in align_words(raw_tokens, ref_tokens):
        if s == g:
            ref_map[pos] = g

    runs, cur = [], []
    for pos, t in enumerate(raw_tokens):
        g = ref_map.get(pos)
        ok = False
        if g is not None and len(t) >= min_len:
            cs = generate_candidates(t, max_candidates=max_candidates, use_vocab=False)
            if g in cs:
                left = raw_tokens[pos - 1] if pos > 0 else "<s>"
                right = raw_tokens[pos + 1] if pos + 1 < len(raw_tokens) else "</s>"
                cur.append({"tok": t, "cands": cs, "gold": cs.index(g),
                            "left": left, "right": right, "pos": pos})
                ok = True
        if not ok:
            if cur:
                runs.append(cur); cur = []
    if cur:
        runs.append(cur)
    return runs, raw_tokens


_ex_runs, _ex_tokens = build_runs_for_file(stage1_df.iloc[0]["raw"], stage1_df.iloc[0]["ref"])
print("Мысал: runs саны =", len(_ex_runs), "| бірінші run ұзындығы =",
      len(_ex_runs[0]) if _ex_runs else 0)

In [60]:
# ---------------------------------------------------------------
# CRFModel: emission = maxent_features (+ BERT err), transition = trans_feats
# ---------------------------------------------------------------
def bert_err_for_file(raw_text, threshold=0.30):
    """Толық мәтіннің әр сөзі үшін BERT детектордың ERR-ықтималдығы."""
    tokens = _core_tokens(raw_text)
    if not tokens:
        return {}
    out = {}
    CH = 200  # BERT токенайзер шегінен аулақ болу үшін сөзбен кесу
    for start in range(0, len(tokens), CH):
        chunk = tokens[start:start + CH]
        _, probs = predict_ok_err(chunk, threshold=threshold)
        for i, p in enumerate(probs):
            out[start + i] = p
    return out


class CRFModel:
    def __init__(self, vec, err_by_pos):
        self.vec = vec
        self.err = err_by_pos          # {pos: prob}
        self.n_em = len(vec.get_feature_names_out()) + 1   # +1 -- bert_err*change
        self.n_tr = N_TR
        self.n = self.n_em + self.n_tr
        self.w = np.zeros(self.n)
        self._cache = {}

    def em_feats(self, s, c, left, right, pos):
        key = (s, c, left, right, pos)
        v = self._cache.get(key)
        if v is None:
            fd = maxent_features(s, c, left, right)
            base = self.vec.transform([fd]).toarray()[0]
            err = self.err.get(pos, 0.0)
            extra = np.array([err * (0.0 if c == s else 1.0)])
            v = np.concatenate([base, extra])
            self._cache[key] = v
        return v

    def trans_feat(self, prev_c, cur_c):
        return trans_feats(prev_c, cur_c)

    def full(self, s, c, left, right, pos, prev_c):
        return np.concatenate([self.em_feats(s, c, left, right, pos),
                               self.trans_feat(prev_c, c)])

print("CRFModel класы дайын.")

CRFModel класы дайын.


In [61]:
# ---------------------------------------------------------------
# CRF: forward-backward оқыту (crf_code/src/crf.py-мен бірдей математика)
# ---------------------------------------------------------------
from scipy.optimize import minimize
from scipy.special import logsumexp as _logsumexp

class CRF:
    def __init__(self, model, l2=1.0):
        self.m = model
        self.l2 = l2
        self.w = np.array(model.w, dtype=float)

    def prepare(self, runs):
        self.data = []
        for run in runs:
            E, T, gold = [], [], []
            for i, st in enumerate(run):
                cs = st["cands"]
                E.append(np.stack([self.m.em_feats(st["tok"], c, st["left"], st["right"], st["pos"])
                                   for c in cs]))
                gold.append(st["gold"])
                if i == 0:
                    prev = [st["left"]]
                else:
                    prev = run[i - 1]["cands"]
                T.append(np.stack([[self.m.trans_feat(p, c) for c in cs] for p in prev]))
            self.data.append((E, T, gold))
        return self

    def _seq(self, E, T, gold, we, wt):
        L = len(E)
        em = [Ei @ we for Ei in E]
        tr = [Ti @ wt for Ti in T]

        alpha = [None] * L
        alpha[0] = em[0] + tr[0][0]
        for t in range(1, L):
            alpha[t] = em[t] + _logsumexp(alpha[t - 1][:, None] + tr[t], axis=0)
        logZ = _logsumexp(alpha[-1])

        beta = [None] * L
        beta[-1] = np.zeros_like(alpha[-1])
        for t in range(L - 2, -1, -1):
            beta[t] = _logsumexp(tr[t + 1] + em[t + 1][None, :] + beta[t + 1][None, :], axis=1)

        score = em[0][gold[0]] + tr[0][0, gold[0]]
        for t in range(1, L):
            score += em[t][gold[t]] + tr[t][gold[t - 1], gold[t]]
        nll = logZ - score

        g_em = np.zeros_like(we)
        g_tr = np.zeros_like(wt)
        for t in range(L):
            p = np.exp(alpha[t] + beta[t] - logZ)
            g_em += E[t].T @ p - E[t][gold[t]]
        for t in range(L):
            if t == 0:
                pj = np.exp(alpha[0] + beta[0] - logZ)
                g_tr += (T[0][0] * pj[:, None]).sum(axis=0) - T[0][0, gold[0]]
            else:
                mm = (alpha[t - 1][:, None] + tr[t] + em[t][None, :] + beta[t][None, :] - logZ)
                pij = np.exp(mm)
                g_tr += (T[t] * pij[:, :, None]).sum(axis=(0, 1)) - T[t][gold[t - 1], gold[t]]
        return nll, g_em, g_tr

    def fit(self, runs, maxiter=150, verbose=True):
        self.prepare(runs)
        if not self.data:
            return self
        n_em = self.m.n_em
        n_tr = self.m.n_tr

        def obj(w):
            we, wt = w[:n_em], w[n_em:]
            tot = 0.0
            gem = np.zeros(n_em); gtr = np.zeros(n_tr)
            for E, T, gold in self.data:
                nll, ge, gt = self._seq(E, T, gold, we, wt)
                tot += nll; gem += ge; gtr += gt
            tot += self.l2 * float(w @ w)
            grad = np.concatenate([gem, gtr]) + 2 * self.l2 * w
            return tot, grad

        r = minimize(obj, self.w, jac=True, method="L-BFGS-B", options={"maxiter": maxiter})
        self.w = r.x
        self.m.w = r.x
        if verbose:
            print("  CRF: %s, итерация %d, NLL %.1f" % (r.message[:40], r.nit, r.fun))
        return self


def crf_decode(model, raw_text, err_by_pos, threshold=0.30, beam=14, max_candidates=30):
    """Бүкіл мәтінді бір тізбек ретінде Viterbi-мен декодтайды.
    ERR деп танылмаған сөздер candidate=[өзі] болып қалады (тізбек үзілмейді)."""
    tokens = _core_tokens(raw_text)
    if not tokens:
        return raw_text
    labels = ["ERR" if err_by_pos.get(i, 0.0) > threshold else "OK" for i in range(len(tokens))]
    we, wt = model.w[:-model.n_tr], model.w[-model.n_tr:]

    T = len(tokens)
    best = [dict() for _ in range(T + 1)]
    best[0]["<s>"] = (0.0, None)

    def trans(prev, c):
        return float(model.trans_feat(prev, c) @ wt)

    for i, t in enumerate(tokens):
        if not best[i]:
            continue
        left = tokens[i - 1] if i > 0 else "<s>"
        right = tokens[i + 1] if i + 1 < T else "</s>"
        if labels[i] == "ERR":
            cs = generate_candidates(t, max_candidates=max_candidates, use_vocab=False)
            if t not in cs:
                cs.append(t)
        else:
            cs = [t]
        em = {c: float(model.em_feats(t, c, left, right, i) @ we) for c in cs}
        if len(cs) > beam:
            cs = sorted(cs, key=lambda c: -em[c])[:beam]
            if t not in cs:
                cs.append(t)
        for c in cs:
            b, arg = -1e18, None
            for prev, (sc, _) in best[i].items():
                v = sc + trans(prev, c)
                if v > b:
                    b, arg = v, prev
            v = b + em[c]
            cur = best[i + 1].get(c)
            if cur is None or v > cur[0]:
                best[i + 1][c] = (v, (i, arg, c))

    end = T
    while end > 0 and not best[end]:
        end -= 1
    cur = max(best[end], key=lambda c: best[end][c][0])
    out_tokens = []
    pos = end
    while pos > 0:
        sc, bp = best[pos][cur]
        i, prev, emitted = bp
        out_tokens.append(emitted)
        pos, cur = i, prev
    out_tokens.reverse()

    # тыныс белгісін бастапқы raw токендерден қайта жабыстырамыз
    raw_full = raw_text.split()
    pre, _, post = [], [], []
    for tk in raw_full:
        p1, _, p2 = _split_punct(tk)
        pre.append(p1); post.append(p2)
    n = min(len(out_tokens), len(pre), len(post))
    rebuilt = [pre[i] + out_tokens[i] + post[i] for i in range(n)]
    rebuilt.extend(out_tokens[n:])
    return " ".join(rebuilt)

print("CRF класы мен Viterbi декодер дайын.")

CRF класы мен Viterbi декодер дайын.


In [63]:
import pickle
CKPT = os.path.join(os.path.dirname(STAGE1_CSV), "crf_checkpoint.pkl")   # үзіліп кетсе, осыдан жалғасады
_ck = {}
if os.path.exists(CKPT):
    try:
        with open(CKPT, "rb") as _f: _ck = pickle.load(_f)
        print("Checkpoint табылды: err_by_file =", "бар" if "err_by_file" in _ck else "жоқ", "| дайын fold-тар:", sorted(_ck.get("folds", {})))
    except Exception as _e:
        print("Checkpoint оқылмады, басынан басталады:", _e); _ck = {}

def _save_ckpt():
    for _m in [v[1] for v in crf_fold_models.values()]:
        _m._cache = {}                      # үлкен кэшті сақтамаймыз
    with open(CKPT, "wb") as _f:
        pickle.dump({"err_by_file": err_by_file, "folds": crf_fold_models}, _f)
    print("   checkpoint сақталды:", CKPT)

# ---------------------------------------------------------------
# BERT err ықтималдықтарын бір рет есептеп аламыз (детектор өзгермейді)
# ---------------------------------------------------------------
err_by_file = _ck.get("err_by_file", {})
if not err_by_file:
    for _, r in tqdm(stage1_df.iterrows(), total=len(stage1_df), desc="BERT err (детектор)"):
        err_by_file[r["stem"]] = bert_err_for_file(r["raw"], threshold=0.30)

# ---------------------------------------------------------------
# 5-fold CV: сол folds (row_to_fold) алдыңғы 3-саты cell-інен қайта
# пайдаланылады -- салыстыру әділ болу үшін бірдей бөліну
# ---------------------------------------------------------------
crf_fold_models = dict(_ck.get("folds", {}))

for f_idx, test_ids in enumerate(folds):
    if f_idx in crf_fold_models:
        print(f"fold {f_idx}: checkpoint-тан алынды"); continue
    test_set = set(test_ids)
    train_ids = [i for i in row_ids if i not in test_set]

    all_runs = []
    for i in train_ids:
        r = stage1_df.iloc[i]
        runs, _ = build_runs_for_file(r["raw"], r["ref"])
        # pos кілтін (stem, pos) ретінде сақтаймыз, бірақ em_feats кэші
        # үшін "pos" осы файлдың ішіндегі позиция болуы жеткілікті, себебі
        # әр CRFModel бір fold-қа бөлек жасалады
        all_runs.extend(runs)

    # осы fold-тың train кандидаттарынан vec (emission feature space) құраймыз
    all_feat_dicts = []
    for run in all_runs:
        for st in run:
            for c in st["cands"]:
                all_feat_dicts.append(maxent_features(st["tok"], c, st["left"], st["right"]))
    vec_crf = DictVectorizer(sparse=True)
    vec_crf.fit(all_feat_dicts)

    # err_by_pos -- бұл fold-тағы барлық train файлдарының бірігуі, бірақ
    # CRFModel-ге бір "combined" сөздік ретінде береміз: кілт = pos ретінде
    # ғана жарамды болу үшін әр файлды БӨЛЕК model-мен оқытамыз -- сондықтан
    # оқытуды файл-файл бойынша емес, барлық runs-ты БІРГЕ (ортақ салмақпен)
    # оқытамыз да, әр файлдың өз err сөздігін decode кезінде береміз.
    model_crf = CRFModel(vec_crf, err_by_pos={})  # оқытуда err кэш пайдаланылмайды (төменде)

    # train кезінде emission-ға bert_err қосу үшін әр run-ның "pos" мәнін
    # (file_idx, pos) деп өзгертеміз, ал err сөздігін сол құрама кілтпен береміз
    combined_err = {}
    all_runs2 = []
    for i in train_ids:
        r = stage1_df.iloc[i]
        runs, _ = build_runs_for_file(r["raw"], r["ref"])
        efile = err_by_file[r["stem"]]
        for run in runs:
            for st in run:
                key = (i, st["pos"])
                combined_err[key] = efile.get(st["pos"], 0.0)
                st["pos"] = key
        all_runs2.extend(runs)
    model_crf.err = combined_err

    crf = CRF(model_crf, l2=1.0).fit(all_runs2, maxiter=150, verbose=True)
    crf_fold_models[f_idx] = (vec_crf, model_crf)
    print(f"fold {f_idx}: train runs саны {len(all_runs2)} дайын")
    _save_ckpt()

print("\n5-fold CRF (морфология+BERT) оқыту аяқталды.")

BERT err (детектор):   0%|          | 0/100 [00:00<?, ?it/s]

NameError: name 'build_runs_for_file' is not defined

In [ ]:
crf_rows = []
for i, (_, r) in enumerate(tqdm(stage1_df.iterrows(), total=len(stage1_df), desc="CRF decode (5-fold)")):
    f_idx = row_to_fold[i]
    vec_crf, model_crf = crf_fold_models[f_idx]
    ref_text = str(r["ref"]); raw_text = str(r["raw"])
    efile = err_by_file[r["stem"]]
    try:
        crf_fixed = crf_decode(model_crf, raw_text, efile, threshold=0.30, beam=14, max_candidates=30)
    except Exception as e:
        print(f"[ERROR] {r['duration']}/{r['stem']}: {e}")
        crf_fixed = raw_text

    crf_rows.append({
        "duration": r["duration"], "stem": r["stem"], "fold": f_idx,
        "ref": ref_text, "raw": raw_text, "crf_morph_bert_fixed": crf_fixed,
        "wer_raw": wer_m(ref_text, raw_text),
        "wer_crf": wer_m(ref_text, crf_fixed),
        "cer_raw": cer_m(ref_text, raw_text),
        "cer_crf": cer_m(ref_text, crf_fixed),
        "bleu_raw": bleu_m(ref_text, raw_text),
        "bleu_crf": bleu_m(ref_text, crf_fixed),
        "chrf_raw": chrf_m(ref_text, raw_text),
        "chrf_crf": chrf_m(ref_text, crf_fixed),
        "ter_raw": ter_m(ref_text, raw_text),
        "ter_crf": ter_m(ref_text, crf_fixed),
        "bertscore_raw": bertscore_m(ref_text, raw_text),
        "bertscore_crf": bertscore_m(ref_text, crf_fixed),
    })

crf_df = pd.DataFrame(crf_rows)
OUT_CSV4 = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage4_crf_morph_bert_results.csv")
crf_df.to_csv(OUT_CSV4, index=False)
print("Сақталды:", OUT_CSV4)
crf_df.head()

In [ ]:
full = crf_df.merge(real_morph_df[["stem", "wer_lexical", "wer_real_morph",
                                    "cer_real_morph", "bleu_real_morph",
                                    "chrf_real_morph", "ter_real_morph",
                                    "bertscore_real_morph"]], on="stem", how="left")
if stage2_df is not None:
    full = full.merge(stage2_df[["stem", "wer_bert", "cer_bert", "bleu_bert",
                                 "chrf_bert", "ter_bert", "bertscore_bert"]],
                      on="stem", how="left")

rows4 = []
for m in ["wer", "cer", "bleu", "chrf", "ter", "bertscore"]:
    row = {"metric": m, "raw": full[f"{m}_raw"].mean()}
    if f"{m}_lexical" in full:
        row["1-саты lexical"] = full[f"{m}_lexical"].mean()
    if f"{m}_bert" in full:
        row["2-саты bert+maxent (синтетика)"] = full[f"{m}_bert"].mean()
    if f"{m}_real_morph" in full:
        row["3-саты bert+maxent+морф (нақты)"] = full[f"{m}_real_morph"].mean()
    row["4-саты bert+морф+CRF (нақты)"] = full[f"{m}_crf"].mean()
    rows4.append(row)

final4_df = pd.DataFrame(rows4).round(4)
display(final4_df)

OUT_XLSX4 = os.path.join(os.path.dirname(STAGE1_CSV), "mapp_stage4_crf_comparison.xlsx")
with pd.ExcelWriter(OUT_XLSX4) as writer:
    full.to_excel(writer, sheet_name="per_file", index=False)
    final4_df.to_excel(writer, sheet_name="summary_comparison", index=False)
print("Excel сақталды:", OUT_XLSX4)

In [ ]:
# ---- БІРЫҢҒАЙ ШЫҒЫС: бір CSV-де raw + үш MAPP нұсқасының мәтіні ----
_base = os.path.dirname(STAGE1_CSV)
_s3 = pd.read_csv(os.path.join(_base, "mapp_stage3_real_morph_maxent_results.csv"))
_s4 = pd.read_csv(os.path.join(_base, "mapp_stage4_crf_morph_bert_results.csv"))
u = _s3[["duration","stem","fold","ref","raw","lexical_fixed","real_morph_fixed"]].merge(
    _s4[["duration","stem","crf_morph_bert_fixed"]], on=["duration","stem"], how="inner")
u = u.rename(columns={"lexical_fixed":"cse_morph","real_morph_fixed":"bert_morph_maxent","crf_morph_bert_fixed":"bert_morph_crf"})
assert len(u)==100, len(u)
print("raw==cse_morph болған файл:", (u.raw==u.cse_morph).sum(), "(0-ге жақын болуы керек)")
OUT_UNI = os.path.join(_base, "mapp_unified_texts_largev3.csv")
u.to_csv(OUT_UNI, index=False); print("Сақталды:", OUT_UNI)